# Investigate Thematic Divergence

This notebook focussed on occurences of semantic divergence between the masked 'machine' token and the 'predictions'. We filtered predictions to only include sentences where the 'machine' is confused with another thematic area. This notebook focusses on the human-machine confusion. 

We there focus in human-type predictions, i.e. occurences where the LMs suggest as human-word as a filler for the machine token. 

For more information on the words we included, inspect `250_freq_pred_KB_edit.txt` and the `1-compute-gradients.ipynb`.


## What can this tell us?

We look at occurences where BERT confused the machine with human words.

We want to understand the divergence, and gauge, which **semantic** and **syntactic** patterns trigger the model towards 'human'-like predictions.

We can look more broadly which words exert push a pull related to (or away from) human or machine agency.

We only look at a smaller subset of examples, based on the existing top 10 predictions recorded in the spreadsheet created by Mariona.

We try to improve and add a few more features:
- a comparative analysis of newspapers and books, i.e. using both newspaper and book data as well as the model trained on these collections
- a more refined syntactic analysis, which includes the syntactic relation of the context token to the masked token

# On Terminology



### A thematic analysis, contrasting two concepts.

We want to understand the semantic intermingling of two distinct concepts, more specifically how the machine are related to, or portrayed as, 'human'-like entities. In this scenario, 'machine' is the TargetMaskedToken (the token that actually appeared in the text) and 'human' is the **predicted contrastive theme**, i.e. a set of thematically related words, predicted by BERT in the position of the masked machine token.

### Target vs. Contrastive Theme

One of the concepts is the **"target concept"**, to other one the **"contrastive concept"**. Each analysis starts with a target concept or token, or `TargetMaskedToken`, this is the starting point of the analysis, for example the word 'machine'. This tokens has two associated datasets. 
    - a TargetMaskedToken maps onto a set of workds that capture the target concept
    - sentences which contain the masked token (`df_target_sent`)
    - contribution obtained by from the integrated gradients method (abbr "ig", `df_target_ig`) 

The `TargetMaskedToken` related to a `ContrastiveTheme`, in this case 'human' like entitites which we filtered from BERT predictions. 

To understand language model predictions, we study both the `TargetMaskedToken` and the `ContrastiveTheme` in two scenarios: **observed** and **counterfactual**. 

 Observed refers to scenarios in which the target of constrastive token appears in the sentences. We than look what tokens contribute to predicting the actual word use. 
 
 In counterfactual scenarios, we measure which words contributed to the ''wrong'' predictions, i.e. what caused BERT to confuse machines with human entities?

## Setting things up and loading the data

In [1]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path

In [7]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence

dataPath = 'input_data' # change to '.' when working in colab 
processedFolder = 'gradient_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

This analysis focuses on 'machine'.


In [21]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t')
print(f'We have {df_sent.shape[0]} sentences for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken}.')


We have 19003 sentences for the target token machine in the blb collection.
We have 1363761 explanations for the target token machine.


In [22]:
# check if ids are aligned between gradient data and sentences
print(df_ig['id'].nunique(), df_sent.shape[0])
print(df_ig[df_ig['id'] == 0].Token, df_sent.iloc[0].currentSentence)

19003 19003
0              "
1           upon
2            the
3         ground
4              -
         ...    
66    fanatacism
67            of
68     crusaders
69             .
70             "
Name: Token, Length: 71, dtype: str " Upon the ground - where they had kneeled, His men would die or win the field, — " and friends and adversaries alike bear testimony, that in his camp, — " the most rigid discipline was found in company with the fiercest resolution; that his troops moved to victory with the precision of machines, while burning with the wildest fanatacism of crusaders. "


In [23]:
df_ig.columns

Index(['Token', 'Score', 'Target', 'id', 'Score_normalized',
       'mask_syntax_relation', 'mask_constituent_relation'],
      dtype='str')

In [26]:
df_ig.Target.unique()

<ArrowStringArray>
[  'soldiers',      'girls',        'men',     'people',      'women',
     'ladies',      'woman',       'girl',        'man',      'child',
 ...
 'individual',     'giants',       'baby',    'peasant',   'subjects',
     'demons',     'angels',      'idiot',    'citizen',   'mistress']
Length: 131, dtype: str

In [27]:
display(
    df_ig[(df_ig.id==9)               
                 ][["id", "Target", "Token",'Score', "Score_normalized", "mask_syntax_relation", "mask_constituent_relation"]]
)

,id,Target,Token,Score,Score_normalized,mask_syntax_relation,mask_constituent_relation
902,9,soldier,braddock,0.101057,0.023027,NaN,nsubj^|conjv|intjv|nsubjv
903,9,soldier,had,0.080576,0.018360,NaN,aux^|conjv|intjv|nsubjv
904,9,soldier,distinguished,0.080049,0.018240,NaN,conjv|intjv|nsubjv
905,9,soldier,himself,0.087602,0.019961,NaN,dobj^|conjv|intjv|nsubjv
906,9,soldier,as,-0.000638,-0.000145,NaN,prep^|conjv|intjv|nsubjv
907,9,soldier,a,0.060755,0.013843,NaN,det^|pobj^|prep^|conjv|intjv|nsubjv
908,9,soldier,tactician,0.189319,0.043138,NaN,pobj^|prep^|conjv|intjv|nsubjv
909,9,soldier,in,0.047165,0.010747,NaN,prep^|pobj^|prep^|conjv|intjv|nsubjv
910,9,soldier,english,0.078509,0.017889,NaN,amod^|pobj^|prep^|pobj^|prep^|conjv|intjv|nsubjv
911,9,soldier,warfare,0.169903,0.038714,NaN,pobj^|prep^|pobj^|prep^|conjv|intjv|nsubjv
